# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
# (The files are only downloaded if they are not here yet.)
![ -f linf_base.pt ] || gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
![ -f cifar10_train_10k.pt ] || gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
![ -f cifar10_test_1k.pt ] || gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: mps


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [5]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = (delta + alpha * grad.sign()).clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        # Random direction, random radius in [0, eps] (same scheme as the l1 attack).
        delta = torch.randn_like(x)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / l2_norm(delta) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)
        delta = delta * (eps / l2_norm(delta).clamp_min(eps))  # scale back only if ||delta||_2 > eps
        delta = clip_box(x, delta)
    return (x + delta).detach()

In [6]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [7]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [8]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

In [9]:
# Helper for the experiments: train from the base model, time each epoch, evaluate, and keep the results.
RESULTS = {}


def run(name, loss_fn, epochs=10, **kwargs):
    start = time.time()
    model = adv_train(load_base_model(), loss_fn, epochs=epochs, **kwargs)
    sec = (time.time() - start) / epochs
    RESULTS[name] = {**test_model(model), "s/epoch": sec}
    return model


def mean_of(name, names):
    RESULTS[name] = {k: float(np.mean([RESULTS[n][k] for n in names])) for k in RESULTS[names[0]]}


def show(names):
    cols = ["clean", "linf", "l2", "l1", "union", "s/epoch"]
    print(f"{'model':<22}" + "".join(f"{c:>9}" for c in cols))
    for n in names:
        r = RESULTS[n]
        print(f"{n:<22}" + "".join(f"{r[c]:9.1f}" for c in cols))

In [10]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
RESULTS["base"] = {**test_model(load_base_model()), "s/epoch": float("nan")}

clean 78.7  linf 48.3  l2 60.0  l1 23.2  union 23.0   (155 s)


# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [11]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [12]:
model_linf = run("linf", single_norm_loss, norm="linf")

epoch 1: loss 1.200 (59 s)


epoch 2: loss 1.174 (118 s)


epoch 3: loss 1.151 (177 s)


epoch 4: loss 1.124 (237 s)


epoch 5: loss 1.096 (296 s)


...

epoch 7: loss 1.045 (414 s)


epoch 8: loss 1.031 (473 s)


epoch 9: loss 1.015 (542 s)


epoch 10: loss 1.008 (608 s)


clean 79.0  linf 44.8  l2 57.6  l1 20.3  union 20.1   (183 s)


In [13]:
model_l2 = run("l2", single_norm_loss, norm="l2")

epoch 1: loss 0.895 (68 s)


epoch 2: loss 0.833 (129 s)


epoch 3: loss 0.791 (195 s)


epoch 4: loss 0.748 (263 s)


epoch 5: loss 0.707 (331 s)


...

epoch 7: loss 0.633 (462 s)


epoch 8: loss 0.611 (527 s)


epoch 9: loss 0.586 (592 s)


epoch 10: loss 0.582 (659 s)


clean 81.6  linf 32.5  l2 59.7  l1 28.6  union 24.5   (170 s)


In [14]:
model_l1 = run("l1", single_norm_loss, norm="l1")

epoch 1: loss 1.228 (70 s)


epoch 2: loss 0.900 (140 s)


epoch 3: loss 0.616 (211 s)


epoch 4: loss 0.468 (281 s)


epoch 5: loss 0.345 (352 s)


...

epoch 7: loss 0.212 (493 s)


epoch 8: loss 0.178 (562 s)


epoch 9: loss 0.151 (633 s)


epoch 10: loss 0.143 (701 s)


clean 82.9  linf 0.0  l2 0.2  l1 32.9  union 0.0   (181 s)


In [15]:
show(["base", "linf", "l2", "l1"])

model                     clean     linf       l2       l1    union  s/epoch
base                       78.7     48.3     60.0     23.2     23.0      nan
linf                       79.0     44.8     57.6     20.3     20.1     60.9
l2                         81.6     32.5     59.7     28.6     24.5     66.0
l1                         82.9      0.0      0.2     32.9      0.0     70.1


### Part (a): answers

All numbers are from `test_model` (1,000 test images, 50 steps, 2 restarts). The training times were measured on an Apple M2 GPU (MPS), not on a T4.

| model | clean | $\ell_\infty$ | $\ell_2$ | $\ell_1$ | union | s/epoch |
|---|---|---|---|---|---|---|
| base | 78.7 | 48.3 | 60.0 | 23.2 | 23.0 | – |
| $\ell_\infty$ AT | 79.0 | 44.8 | 57.6 | 20.3 | 20.1 | 61 |
| $\ell_2$ AT | 81.6 | 32.5 | 59.7 | 28.6 | 24.5 | 66 |
| $\ell_1$ AT | 82.9 | 0.0 | 0.2 | 32.9 | 0.0 | 70 |

**(i) Weakest norm.** Base model: $\ell_1$ (23.2%). $\ell_\infty$ model: $\ell_1$ (20.3%). $\ell_2$ model: $\ell_1$ (28.6%), with $\ell_\infty$ close behind (32.5%). $\ell_1$ model: $\ell_\infty$ (0.0%), and $\ell_2$ is almost as bad (0.2%).
For every model, the union accuracy is almost equal to the accuracy for the weakest norm.

**(ii) Transfer.** Robustness transfers only between norms whose perturbation regions are similar, and it does not transfer between $\ell_\infty$ and $\ell_1$.
* $\ell_\infty \to \ell_2$ transfers well. The base and $\ell_\infty$ models were never trained against $\ell_2$, but they have 58–60% $\ell_2$ accuracy, as high as the $\ell_2$ model (59.7%). An $\ell_2$ perturbation of norm 0.5 spread over many pixels changes each pixel by a small amount, like an $\ell_\infty$ perturbation.
* $\ell_\infty \to \ell_1$ does not transfer: 23.2% and 20.3%. An $\ell_1$ perturbation of norm 12 can change a few pixels by up to 1.0, far outside the $\ell_\infty$ box of 8/255.
* $\ell_2$ is in between. $\ell_2$ training gains 5.4 points of $\ell_1$ accuracy (23.2 → 28.6), but loses 15.8 points of $\ell_\infty$ accuracy (48.3 → 32.5).
* $\ell_1 \to$ others gives no transfer at all, and it even removes the robustness of the base model. Ten epochs of $\ell_1$ fine-tuning take $\ell_\infty$ accuracy from 48.3% to 0.0% and $\ell_2$ from 60.0% to 0.2%. The $\ell_1$ model also has the highest clean accuracy, so it trades dense-perturbation robustness for clean accuracy and sparse-perturbation robustness.
* The $\ell_1$ model only reaches 32.9% $\ell_1$ accuracy, although its training loss goes down to 0.14 (the $\ell_\infty$ model ends at 1.01). A likely reason is that the 10-step training attack changes at most $10 \cdot 20 = 200$ coordinates, so it is much weaker than the 50-step test attack. The model fits this weak attack, and this does not give robustness against the strong one.

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [16]:
def ramp_loss(model, x, y, lam=0.5):
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    logits_q, logits_r = model(x_q), model(x_r)
    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    # Logit pairing KL(p_q || p_r), only on the images that are correct under the l_q attack
    # (summed over these images and divided by the batch size, as in the RAMP code).
    correct_q = (logits_q.argmax(1) == y).float()
    kl = F.kl_div(F.log_softmax(logits_r, dim=1), F.softmax(logits_q, dim=1), reduction="none").sum(1)
    return torch.max(ce_q, ce_r).mean() + lam * (kl * correct_q).sum() / len(y)


def gradient_projection(g_n, g_a):
    cos = F.cosine_similarity(g_n.flatten(), g_a.flatten(), dim=0)
    return cos.clamp_min(0) * g_n  # cos * g_n if cos > 0, else 0

In [17]:
model_ramp = run("ramp (seed 0)", ramp_loss, gp_beta=0.5)

epoch 1: loss 1.416 (167 s)


epoch 2: loss 1.327 (317 s)


epoch 3: loss 1.298 (466 s)


epoch 4: loss 1.270 (598 s)


epoch 5: loss 1.236 (727 s)


...

epoch 7: loss 1.194 (986 s)


epoch 8: loss 1.183 (1115 s)


epoch 9: loss 1.162 (1245 s)


epoch 10: loss 1.161 (1375 s)


clean 78.5  linf 42.6  l2 61.5  l1 45.4  union 39.9   (154 s)


In [18]:
show(["base", "linf", "l2", "l1", "ramp (seed 0)"])

model                     clean     linf       l2       l1    union  s/epoch
base                       78.7     48.3     60.0     23.2     23.0      nan
linf                       79.0     44.8     57.6     20.3     20.1     60.9
l2                         81.6     32.5     59.7     28.6     24.5     66.0
l1                         82.9      0.0      0.2     32.9      0.0     70.1
ramp (seed 0)              78.5     42.6     61.5     45.4     39.9    137.5


### Part (b): answers

| model | clean | $\ell_\infty$ | $\ell_2$ | $\ell_1$ | union | s/epoch |
|---|---|---|---|---|---|---|
| base | 78.7 | 48.3 | 60.0 | 23.2 | 23.0 | – |
| $\ell_\infty$ AT | 79.0 | 44.8 | 57.6 | 20.3 | 20.1 | 61 |
| $\ell_2$ AT | 81.6 | 32.5 | 59.7 | 28.6 | 24.5 | 66 |
| $\ell_1$ AT | 82.9 | 0.0 | 0.2 | 32.9 | 0.0 | 70 |
| RAMP (seed 0) | 78.5 | 42.6 | 61.5 | 45.4 | **39.9** | 138 |

Training time for each RAMP epoch (seed 0), from the cumulative times in the log: 167, 150, 149, 132, 129, 129, 130, 129, 130, 130 s.
The first epoch is slower because of warm-up.

**(i)** RAMP gives the highest union accuracy: 39.9%. The best single-norm model is the $\ell_2$ model with 24.5%, and the base model has 23.0%.
RAMP almost doubles the $\ell_1$ accuracy of the base model (23.2 → 45.4). It loses only 5.7 points of $\ell_\infty$ (48.3 → 42.6) and gains 1.5 points of $\ell_2$, so no norm is far below the others.
The $\ell_1$ model gives the highest clean accuracy (82.9%), but its union accuracy is 0. RAMP keeps the clean accuracy of the base model (78.5% vs 78.7%).

**(ii)** RAMP uses 138 s per epoch, against 61–70 s for single-norm training (mean 66 s): about **2.1×** as much time.
* Most of the cost of adversarial training is the attack. A 10-step attack is 10 forward and backward passes, while the weight update is only 1.
* RAMP runs two attacks for each batch ($\ell_\infty$ and $\ell_1$), so the attack cost doubles.
* In addition, RAMP does two training forward passes (on $x^q$ and $x^r$) instead of one. It also does an extra forward and backward pass on the clean images for $g_n$, and calculates the cosine similarity and projection for each parameter tensor.

The RAMP run with seed 1 (part (c)) measured 181 s per epoch, because other jobs were running on the same machine at that time. I use seed 0 for the time comparison.

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

### (i) Idea: adaptive norm allocation with teacher anchoring

**Method.** For each batch:
1. **One attack per image.** Each training image gets exactly one attack. Its norm $k \in \{\ell_\infty, \ell_2, \ell_1\}$ is sampled with probability $s_k$.
2. **Adaptive allocation.** $s_k$ is proportional to $e_k$, an exponential moving average (momentum 0.9) of the model's training error on the images that were attacked with norm $k$. A floor of 10% of the mass is spread equally over the norms, so no norm gets less than 3.3%. The norm where the model currently fails most gets the most training images.
3. **Teacher anchor.** For the images attacked with $\ell_\infty$, the predictions of the model are kept close to the predictions of the frozen base model $f_0$. This is done only on the images that the base model classifies correctly. The base model is a strong $\ell_\infty$ model, and the anchor is meant to stop the fine-tuning from forgetting this robustness.

The loss is
$$\mathcal{L} = \frac{1}{n}\sum_{i} \mathcal{L}_{CE}\big(f_\theta(x_i^{k_i}), y_i\big) \;+\; \gamma\,\frac{1}{n}\sum_{i:\,k_i=\infty,\ f_0(x_i^{\infty}) = y_i} KL\big(f_0(x_i^{\infty}) \,\|\, f_\theta(x_i^{\infty})\big), \qquad k_i \sim s,\ \ \gamma = 1.$$

Pseudocode for one step:
```
s = 0.1/3 + 0.9 * e / sum(e)                  # share of each norm
k_i ~ Categorical(s) for each image i
x_adv_i = attack_{k_i}(x_i)   (model in eval mode, 10 steps)
loss = CE(f(x_adv), y) + gamma * KL(f0(x_adv) || f(x_adv)) on {k_i = linf, f0 correct}
e_k = 0.9 e_k + 0.1 * error of f on {k_i = k}  # update the EMA
```

**Why I expect it to increase the union accuracy.**
* The union accuracy is limited by the weakest norm: $\ell_1$ for the base model. Part (a) shows that training on one norm does not transfer to $\ell_1$. Every batch must contain all the norms, and more of the norm that currently fails.
* MAX and RAMP attack every image with several norms, so each step costs 2–3 attacks. With one attack per image, each step costs the same as single-norm training. Over the epochs, the model still sees all three norms.
* Part (a) shows that fine-tuning on another norm quickly removes the $\ell_\infty$ robustness of the base model ($\ell_1$ training takes $\ell_\infty$ to 0%). The anchor costs one forward pass of $f_0$ without gradients, and it keeps the $\ell_\infty$ knowledge of the base model.

**Related work and what is new.**
* E-AT (Croce & Hein, *Adversarial robustness against multiple and single $\ell_p$-threat models via quick fine-tuning of robust classifiers*, ICML 2022) also fine-tunes a robust model with one attack per batch. It selects $\ell_\infty$ or $\ell_1$ for each batch with fixed, equal probability. In my method, the norm is selected for each image, over all three norms, and the probabilities adapt to the current error of the model.
* Robust distillation (ARD, Goldblum et al., *Adversarially robust distillation*, AAAI 2020) distills a robust teacher into a student for the same threat model. My method uses the starting model itself as the teacher, only for the norm it is already robust against, and only on the images it classifies correctly. The goal is to prevent forgetting during multi-norm fine-tuning, not to compress a model.
* The method is not MAX or AVG, because each image is attacked once. It is not TRADES, because the KL term pairs the model with the frozen base model, not with its own clean predictions.

In [19]:
# Adaptive norm allocation + teacher anchoring (see the description above).
TEACHER = load_base_model().eval()  # frozen copy of the base model
for p in TEACHER.parameters():
    p.requires_grad_(False)
LAST_LOSS = None


def make_my_loss(anchor=1.0, adaptive=True, momentum=0.9, floor=0.1):
    err = torch.full((len(NORMS),), 0.5)  # EMA of the training error under each attack

    def my_loss(model, x, y):
        n = len(y)
        # Share of the batch for each norm: proportional to its current error, with a minimum share floor / 3.
        share = err / err.sum() if adaptive else torch.full((len(NORMS),), 1 / len(NORMS))
        share = floor / len(NORMS) + (1 - floor) * share
        assign = torch.multinomial(share, n, replacement=True).to(x.device)

        # One attack per image (the cost of single-norm training).
        model.eval()
        x_adv = x.clone()
        for i, norm in enumerate(NORMS):
            m = assign == i
            if m.any():
                x_adv[m] = attack(model, x[m], y[m], norm)
        model.train()
        logits = model(x_adv)
        loss = F.cross_entropy(logits, y)

        # Anchor: on the linf examples, keep the predictions of the base model where the base model is correct.
        if anchor > 0:
            m = assign == NORMS.index("linf")
            with torch.no_grad():
                p_teacher = F.softmax(TEACHER(x_adv[m]), dim=1)
                ok = (p_teacher.argmax(1) == y[m]).float()
            kl = F.kl_div(F.log_softmax(logits[m], dim=1), p_teacher, reduction="none").sum(1)
            loss = loss + anchor * (kl * ok).sum() / n

        with torch.no_grad():
            wrong = (logits.argmax(1) != y).float().cpu()
            a = assign.cpu()
            for i in range(len(NORMS)):
                if (a == i).any():
                    err[i] = momentum * err[i] + (1 - momentum) * wrong[a == i].mean()
        my_loss.history.append(share.tolist() + err.tolist())
        return loss

    global LAST_LOSS
    my_loss.history = []
    LAST_LOSS = my_loss
    return my_loss

In [20]:
# (iii) Results: my method and RAMP, seeds 0 and 1.
model_mine = run("mine (seed 0)", make_my_loss())
alloc_mine = LAST_LOSS.history

epoch 1: loss 1.220 (69 s)


epoch 2: loss 1.158 (135 s)


epoch 3: loss 1.126 (201 s)


epoch 4: loss 1.092 (266 s)


epoch 5: loss 1.057 (332 s)


...

epoch 7: loss 1.007 (463 s)


epoch 8: loss 0.978 (528 s)


epoch 9: loss 0.978 (594 s)


epoch 10: loss 0.962 (661 s)


clean 78.4  linf 41.6  l2 61.5  l1 47.4  union 39.7   (155 s)


In [21]:
model_mine_s1 = run("mine (seed 1)", make_my_loss(), seed=1)

epoch 1: loss 1.232 (66 s)


epoch 2: loss 1.164 (132 s)


epoch 3: loss 1.132 (202 s)


epoch 4: loss 1.091 (274 s)


epoch 5: loss 1.068 (348 s)


...

epoch 7: loss 1.006 (500 s)


epoch 8: loss 0.990 (576 s)


epoch 9: loss 0.981 (660 s)


epoch 10: loss 0.966 (736 s)


clean 78.2  linf 41.5  l2 62.4  l1 47.0  union 39.5   (191 s)


In [22]:
model_ramp_s1 = run("ramp (seed 1)", ramp_loss, gp_beta=0.5, seed=1)

epoch 1: loss 1.421 (167 s)


epoch 2: loss 1.330 (370 s)


epoch 3: loss 1.289 (505 s)


epoch 4: loss 1.267 (676 s)


epoch 5: loss 1.246 (847 s)


...

epoch 7: loss 1.200 (1193 s)


epoch 8: loss 1.174 (1404 s)


epoch 9: loss 1.168 (1624 s)


epoch 10: loss 1.163 (1807 s)


clean 78.7  linf 41.5  l2 61.8  l1 44.6  union 39.4   (196 s)


In [23]:
mean_of("ramp (mean)", ["ramp (seed 0)", "ramp (seed 1)"])
mean_of("mine (mean)", ["mine (seed 0)", "mine (seed 1)"])
show(["base", "ramp (seed 0)", "ramp (seed 1)", "ramp (mean)", "mine (seed 0)", "mine (seed 1)", "mine (mean)"])

model                     clean     linf       l2       l1    union  s/epoch
base                       78.7     48.3     60.0     23.2     23.0      nan
ramp (seed 0)              78.5     42.6     61.5     45.4     39.9    137.5
ramp (seed 1)              78.7     41.5     61.8     44.6     39.4    180.7
ramp (mean)                78.6     42.0     61.6     45.0     39.6    159.1
mine (seed 0)              78.4     41.6     61.5     47.4     39.7     66.1
mine (seed 1)              78.2     41.5     62.4     47.0     39.5     73.6
mine (mean)                78.3     41.5     62.0     47.2     39.6     69.8


In [24]:
# (iv) Ablation (seed 0): remove the teacher anchor, or replace the adaptive allocation by a uniform one.
run("mine, no anchor", make_my_loss(anchor=0.0))
alloc_no_anchor = LAST_LOSS.history
run("mine, uniform alloc", make_my_loss(adaptive=False))
show(["mine (seed 0)", "mine, no anchor", "mine, uniform alloc"])

epoch 1: loss 1.211 (88 s)


epoch 2: loss 1.146 (176 s)


epoch 3: loss 1.105 (249 s)


epoch 4: loss 1.073 (331 s)


epoch 5: loss 1.045 (411 s)


...

epoch 7: loss 0.975 (523 s)


epoch 8: loss 0.957 (594 s)


epoch 9: loss 0.935 (664 s)


epoch 10: loss 0.934 (735 s)


clean 79.1  linf 41.7  l2 62.5  l1 48.5  union 40.3   (171 s)
model                     clean     linf       l2       l1    union  s/epoch
mine (seed 0)              78.4     41.6     61.5     47.4     39.7     66.1
mine, no anchor            78.7     40.7     61.5     47.4     38.9     78.8
mine, uniform alloc        79.1     41.7     62.5     48.5     40.3     73.5


In [25]:
# (v) Analysis: the share of each norm in the batches and the EMA of the training error, per epoch.
def per_epoch(history, steps_per_epoch=len(x_train) // 128):
    h = np.array(history)  # columns: share linf, l2, l1, then EMA error linf, l2, l1
    return h.reshape(-1, steps_per_epoch, h.shape[1]).mean(1)


for name, hist in [("mine (seed 0)", alloc_mine), ("mine, no anchor", alloc_no_anchor)]:
    print(name)
    print("epoch  share(linf, l2, l1)    EMA train error(linf, l2, l1)")
    for e, row in enumerate(per_epoch(hist)):
        print(f"{e + 1:5d}  {row[0]:.2f} {row[1]:.2f} {row[2]:.2f}        {row[3]:.2f} {row[4]:.2f} {row[5]:.2f}")

mine (seed 0)
epoch  share(linf, l2, l1)    EMA train error(linf, l2, l1)
    1  0.36 0.27 0.37        0.50 0.36 0.52
    2  0.38 0.27 0.36        0.49 0.34 0.47
    3  0.39 0.25 0.36        0.49 0.31 0.45
    4  0.38 0.26 0.36        0.47 0.30 0.44
    5  0.39 0.26 0.35        0.46 0.29 0.42
    6  0.41 0.25 0.35        0.47 0.27 0.40
    7  0.39 0.26 0.35        0.45 0.28 0.40
    8  0.40 0.25 0.35        0.45 0.26 0.38
    9  0.41 0.25 0.34        0.44 0.26 0.37
   10  0.40 0.25 0.35        0.44 0.26 0.37
mine, no anchor
epoch  share(linf, l2, l1)    EMA train error(linf, l2, l1)
    1  0.35 0.27 0.38        0.49 0.36 0.53
    2  0.38 0.26 0.36        0.50 0.33 0.47
    3  0.38 0.26 0.36        0.47 0.31 0.45
    4  0.39 0.26 0.35        0.49 0.30 0.44
    5  0.40 0.25 0.35        0.47 0.29 0.41
    6  0.40 0.25 0.36        0.46 0.27 0.41
    7  0.40 0.25 0.35        0.46 0.27 0.39
    8  0.40 0.24 0.35        0.46 0.26 0.39
    9  0.42 0.23 0.35        0.46 0.23 0.37
   10  0.40 0.

In [26]:
# (vi) Check the evaluation: the best model of my method with stronger attacks.
best = max(["mine (seed 0)", "mine (seed 1)"], key=lambda n: RESULTS[n]["union"])
best_model = model_mine if best == "mine (seed 0)" else model_mine_s1
print(best, "default eval: ", "  ".join(f"{k} {RESULTS[best][k]:.1f}" for k in ("clean", *NORMS, "union")))
RESULTS[best + " strong"] = {**test_model(best_model, steps=200, restarts=5), "s/epoch": float("nan")}

mine (seed 0) default eval:  clean 78.4  linf 41.6  l2 61.5  l1 47.4  union 39.7


clean 78.4  linf 41.5  l2 61.2  l1 46.1  union 39.2   (1818 s)


### (iii) Results

Mean of seed 0 and seed 1. The times were measured on an Apple M2 GPU, and the training was 10 epochs with the default settings of `adv_train`.

| method | clean | $\ell_\infty$ | $\ell_2$ | $\ell_1$ | union | s/epoch |
|---|---|---|---|---|---|---|
| base | 78.7 | 48.3 | 60.0 | 23.2 | 23.0 | – |
| RAMP (seed 0 / seed 1) | 78.5 / 78.7 | 42.6 / 41.5 | 61.5 / 61.8 | 45.4 / 44.6 | 39.9 / 39.4 | 138 / 181* |
| **RAMP (mean)** | 78.6 | **42.0** | 61.6 | 45.0 | **39.6** | 138 |
| mine (seed 0 / seed 1) | 78.4 / 78.2 | 41.6 / 41.5 | 61.5 / 62.4 | 47.4 / 47.0 | 39.7 / 39.5 | 66 / 74 |
| **mine (mean)** | 78.3 | 41.5 | **62.0** | **47.2** | **39.6** | **70** |

\*Other jobs were running on the machine during the RAMP seed-1 run, so I use the seed-0 time (138 s) for RAMP.

My method has the **same union accuracy as RAMP (39.6%) with about half the training time per epoch** (70 s vs 138 s).
It has a higher $\ell_1$ accuracy (+2.2) and $\ell_2$ accuracy (+0.4), and a slightly lower $\ell_\infty$ accuracy (−0.5) and clean accuracy (−0.3).
My method uses less time than RAMP, so no comparison with equal training time is needed.

### (iv) Ablation (seed 0)

| variant | clean | $\ell_\infty$ | $\ell_2$ | $\ell_1$ | union | s/epoch |
|---|---|---|---|---|---|---|
| mine (full) | 78.4 | 41.6 | 61.5 | 47.4 | 39.7 | 66 |
| without the teacher anchor ($\gamma = 0$) | 78.7 | 40.7 | 61.5 | 47.4 | 38.9 | 79 |
| uniform allocation ($s_k = 1/3$) | 79.1 | 41.7 | 62.5 | 48.5 | **40.3** | 74 |

The time differences between these runs (66–79 s) come from the load on the machine. The anchor adds only one forward pass without gradients.

### (v) Analysis

**Overall: partly.** The main goal works: one attack per image over all three norms matches RAMP's union accuracy at about half the cost. Of the two extra parts, the anchor helps a little, and the adaptive allocation does not help.

* **Which norm limits the union.** For the base model, $\ell_1$ limits the union (23.2% vs 48.3% for $\ell_\infty$). After my method, $\ell_1$ is at 47.2% and $\ell_\infty$ is at 41.5%, so $\ell_\infty$ is now the weakest norm. The union (39.6%) is close to the $\ell_\infty$ accuracy. To increase the union further, the method must lose less $\ell_\infty$ robustness.
* **Teacher anchor: works as intended, small effect.** Without the anchor, the $\ell_\infty$ accuracy goes down from 41.6% to 40.7%, while $\ell_1$ and $\ell_2$ do not change (47.4 and 61.5). So the union goes down from 39.7% to 38.9%. The anchor protects exactly the norm it is applied to, as designed. The effect (0.8 points) is larger than the seed-to-seed difference of my method (0.2 points), but it is from one seed.
* **Adaptive allocation: does not help.** The uniform allocation gives 40.3% union against 39.7%. The allocation log of cell (v) explains why:
  * The shares stay close to uniform for the whole run: $(\ell_\infty, \ell_2, \ell_1) = (0.36, 0.27, 0.37)$ in epoch 1 and $(0.40, 0.25, 0.35)$ in epoch 10. This is because the training errors under the 10-step attacks are similar for $\ell_\infty$ and $\ell_1$: 0.50 and 0.52 at the start, 0.44 and 0.37 at the end. A share proportional to the error then changes the allocation only a little.
  * Compared with the uniform allocation, the adaptive rule gives more images to $\ell_\infty$ (0.40 vs 0.33), about the same to $\ell_1$ (0.35 vs 0.33), and fewer to $\ell_2$ (0.25 vs 0.33). It does this because $\ell_\infty$ has the highest training error and $\ell_2$ the lowest.
  * The extra $\ell_\infty$ images did not increase the $\ell_\infty$ accuracy (41.6 vs 41.7). The fewer $\ell_2$ images cost accuracy for $\ell_2$ and also for $\ell_1$ (61.5 / 47.4 vs 62.5 / 48.5).
  * A likely reason: $\ell_2$ examples are between $\ell_\infty$ and $\ell_1$ (dense, but with larger changes for some pixels), so training on them helps against both. The error-proportional rule gives few images to $\ell_2$ because it is the easiest norm, and it ignores this transfer.
  * The difference of 0.6 points is close to the seed-to-seed spread (RAMP: 39.9 vs 39.4, mine: 39.7 vs 39.5), so the allocation is at best neutral.
* **Training loss.** My method ends at a loss of about 0.96, and RAMP at 1.16. RAMP's loss is larger because it takes the maximum over two attacks for each image, and my loss uses one attack per image. The training loss of my method decreases steadily, with no sign of instability.
* **What I would change.** A better allocation signal would be the error under a stronger attack measured on a few held-out batches, or a per-image choice of the norm with the highest loss (this would cost more attacks). The results also suggest that the cheap mixing of norms is where most of the gain comes from.

### (vi) Check the evaluation

The best model of my method (seed 0) with `test_model(model, steps=200, restarts=5)`:

| evaluation | clean | $\ell_\infty$ | $\ell_2$ | $\ell_1$ | union |
|---|---|---|---|---|---|
| default (50 steps, 2 restarts) | 78.4 | 41.6 | 61.5 | 47.4 | 39.7 |
| strong (200 steps, 5 restarts) | 78.4 | 41.5 | 61.2 | 46.1 | 39.2 |

The stronger attacks lower the accuracy only a little: −0.1 for $\ell_\infty$, −0.3 for $\ell_2$, −1.3 for $\ell_1$, and −0.5 for union.
The largest drop is for $\ell_1$, which is expected: the sparse $\ell_1$ attack changes only 20 coordinates per step, so it gains the most from more steps and restarts.
If the model masked gradients, the stronger attacks would lower the accuracy much more. So there is no sign of gradient masking.
My method does not change the model at test time (no randomness and no input transformation), so an adaptive attack is not needed.